# 09A SenseVoice 混合噪音增强微调 — 混合A（原始 + 高斯合成噪音）

**混合数据策略**：原始干净数据 + 高斯合成噪音增强数据，1:1 混合

数据量：2x 原始数据（原始 + 噪音A 各50%）

## 1. 环境检查

In [ ]:
import torch
import subprocess
import json
import os
import random
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. 数据准备

In [ ]:
import torch
import soundfile as sf
import json
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

RAW_TRAIN = "/mnt/workspace/data/prepared_data/sensevoice/train.jsonl"
VAL_JSONL = "/mnt/workspace/data/prepared_data/sensevoice/val.jsonl"
PREPARED_DIR = "/mnt/workspace/data/prepared_data/sv_ablation"
os.makedirs(PREPARED_DIR, exist_ok=True)

def load_jsonl(path):
    data = []
    with open(path) as f:
        for line in f:
            data.append(json.loads(line))
    return data

raw_train = load_jsonl(RAW_TRAIN)
print(f"原始训练集: {len(raw_train)} 条")

def add_gaussian_noise(audio_path, snr_db_range=(10, 20)):
    wav, sr = sf.read(audio_path)
    wav = torch.from_numpy(wav).float()
    snr_db = random.uniform(*snr_db_range)
    signal_power = (wav ** 2).mean()
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = torch.randn_like(wav) * torch.sqrt(noise_power)
    return (wav + noise).numpy(), sr

def generate_noise_a_only(data, output_dir, output_filename="train_noise_a_only.jsonl"):
    noise_dir = os.path.join(output_dir, "noise_a_only")
    os.makedirs(noise_dir, exist_ok=True)
    jsonl_path = os.path.join(output_dir, output_filename)
    new_data = []
    for item in data:
        src = item['source']
        try:
            noisy_wav, sr = add_gaussian_noise(src)
            base = os.path.splitext(os.path.basename(src))[0]
            noisy_path = os.path.join(noise_dir, f"{base}_noise_a.wav")
            sf.write(noisy_path, noisy_wav, sr)
            new_item = item.copy()
            new_item['source'] = noisy_path
            new_data.append(new_item)
        except Exception as e:
            print(f"  噪音处理失败: {src}, {e}")
    with open(jsonl_path, 'w') as f:
        for item in new_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"  纯噪音A: {len(new_data)} 条")
    return jsonl_path

def mix_jsonl(jsonl_a, jsonl_b, output_path):
    data_a = load_jsonl(jsonl_a)
    data_b = load_jsonl(jsonl_b)
    mixed = []
    for item_a, item_b in zip(data_a, data_b):
        mixed.append(item_a)
        mixed.append(item_b)
    with open(output_path, 'w') as f:
        for item in mixed:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"  混合数据: {len(mixed)} 条 (A:{len(data_a)} + B:{len(data_b)})")
    return output_path

## 3. 生成混合数据（原始 + 噪音A）

In [ ]:
print("=" * 60)
print("生成混合A数据（原始 + 高斯合成噪音）")
print("=" * 60)

# 噪音A数据
noise_a_only_path = generate_noise_a_only(
    raw_train,
    PREPARED_DIR,
    "train_noise_a_only.jsonl"
)

# 原始数据 jsonl
raw_jsonl = os.path.join(PREPARED_DIR, "train_raw.jsonl")
with open(raw_jsonl, 'w') as f:
    for item in raw_train:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
print(f"  原始数据: {len(raw_train)} 条")

# 混合A = 原始 + 噪音A（1:1）
mixed_a_path = os.path.join(PREPARED_DIR, "train_mixed_a.jsonl")
mix_jsonl(raw_jsonl, noise_a_only_path, mixed_a_path)

print(f"\n混合A数据已就绪: {mixed_a_path}")

## 4. 训练混合A

In [ ]:
import subprocess
import sys

def train_sensevoice_lora(train_jsonl, output_dir, exp_name, port=29500):
    os.makedirs(output_dir, exist_ok=True)
    cmd = [
        "torchrun", f"--nproc_per_node=1", f"--rdzv_endpoint=localhost:{port}",
        "-m", "funasr.bin.train_ds",
        "++model=iic/SenseVoiceSmall",
        f"++train_data_set_list={train_jsonl}",
        f"++valid_data_set_list={VAL_JSONL}",
        "++dataset_conf.data_split_num=1",
        "++dataset_conf.batch_sampler=BatchSampler",
        "++dataset_conf.batch_size=6000",
        "++dataset_conf.sort_size=1024",
        "++dataset_conf.batch_type=token",
        "++dataset_conf.num_workers=4",
        "++train_conf.max_epoch=10",
        "++train_conf.log_interval=50",
        "++train_conf.validate_interval=2000",
        "++train_conf.save_checkpoint_interval=2000",
        "++train_conf.keep_nbest_models=5",
        "++train_conf.avg_nbest_model=3",
        "++train_conf.use_bf16=true",
        "++train_conf.grad_clip=5",
        "++lora_only=true",
        "++lora_bias=none",
        "++lora_rank=8",
        "++lora_alpha=16",
        "++lora_dropout=0.1",
        "++optim=adamw",
        "++optim_conf.lr=2e-4",
        "++optim_conf.weight_decay=0.0",
        f"++output_dir={output_dir}",
    ]
    print(f"\n{'='*60}")
    print(f"训练: {exp_name}")
    print(f"数据: {train_jsonl}")
    print(f"输出: {output_dir}")
    print(f"{'='*60}")
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
        sys.stdout.flush()
    returncode = process.wait()
    if returncode == 0:
        print(f"✓ {exp_name} 训练完成!")
    else:
        print(f"✗ {exp_name} 训练失败! 返回码: {returncode}")
    return returncode == 0

ckpt_path = "/mnt/output/sv_mixed_a/model.pt.best"
if os.path.exists(ckpt_path):
    print("\n[跳过] 混合A 已完成，checkpoint 存在")
else:
    success = train_sensevoice_lora(
        mixed_a_path,
        "/mnt/output/sv_mixed_a",
        "混合A (原始+高斯合成噪音)",
        port=29502,
    )
    print(f"\n训练结果: {'✓ 成功' if success else '✗ 失败'}")